In [1]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-user-shield" style="color:#18ab4b"></i> Linear fraud classifier — SageMaker local mode

Train a <i class="fas fa-bolt" style="color:#6c63ff"></i> **linear SVM** (hinge loss, balanced classes) on synthetic credit-card transactions with SageMaker **local mode** (`instance_type='local'` — a real Docker training job in a bring-your-own-container), evaluate precision/recall, and store the <i class="fas fa-brain" style="color:#18ab4b"></i> model in oblako's <i class="fas fa-database" style="color:#18ab4b"></i> **S3Proxy**.

Same model the original managed-LinearLearner notebook used (`loss=hinge_loss`, `positive_example_weight_mult=balanced`) — here it runs anywhere, no AWS account.

Prereqs: `oblako up s3proxy`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

## <i class="fas fa-file-code" style="color:#6c63ff"></i> Build the container + <i class="fas fa-user-graduate" style="color:#18ab4b"></i> train

In [2]:
import os
import tempfile

import frauddata
from sagemaker.estimator import Estimator

from oblako.services import SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")

IMAGE = "oblako-fraud:latest"
HP = {"epochs": "30"}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)  # linear-SVM BYOC (./container)

# Synthetic creditcard.csv-shaped data: Time, V1..V28, Amount, Class (1 = fraud).
data_dir = tempfile.mkdtemp(prefix="fraud-train-")
train_csv = os.path.join(data_dir, "train.csv")
frauddata.write_training_csv(train_csv, n=4000, seed=0)
test_csv = os.path.join(tempfile.mkdtemp(prefix="fraud-test-"), "test.csv")
frauddata.write_training_csv(test_csv, n=1500, seed=7)
out_dir = tempfile.mkdtemp(prefix="fraud-out-")
print(f"train fraud rate: {frauddata.fraud_rate(train_csv) * 100:.1f}%")

estimator = Estimator(
    image_uri=IMAGE,
    role="arn:aws:iam::000000000000:role/dummy",
    instance_count=1,
    instance_type="local",
    sagemaker_session=sm.get_session(),
    output_path=f"file://{out_dir}",
    hyperparameters=HP,
)
estimator.fit({"train": f"file://{data_dir}"})

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/deburky/Library/Application Support/sagemaker/config.yaml


Step 1/6 : FROM python:3.11-slim
 ---> a3ab0b966bc4
Step 2/6 : RUN pip install --no-cache-dir scikit-learn pandas joblib
 ---> Running in 3981c9092627
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 113.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
 ---> 3646ff8a8ba5
Step 3/6 : COPY src/ /opt/program/
 ---> b2d58eb157b8
Step 4/6 : WORKDIR /opt/p

INFO:sagemaker:Creating training-job with name: oblako-fraud-2026-05-30-14-43-15-140


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


...

time="2026-05-30T16:43:15+02:00" level=warning msg="/private/tmp/claude-501/tmpthirf8d6/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-05-30T16:43:15+02:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpthirf8d6\".\nSet `external: true` to use an existing network"
 Container fy0kahqxkw-algo-1-tg2wl Creating 
 Container fy0kahqxkw-algo-1-tg2wl Created 
Attaching to fy0kahqxkw-algo-1-tg2wl
 Container fy0kahqxkw-algo-1-tg2wl Starting 


 Container fy0kahqxkw-algo-1-tg2wl Started 


fy0kahqxkw-algo-1-tg2wl  | /usr/local/lib/python3.11/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
fy0kahqxkw-algo-1-tg2wl  |   warnings.warn(
fy0kahqxkw-algo-1-tg2wl  | trained on 4000 rows, 19.7% fraud, 30 epochs
fy0kahqxkw-algo-1-tg2wl  | saved model to /opt/ml/model


INFO:sagemaker.local.image:===== Job Complete =====


fy0kahqxkw-algo-1-tg2wl exited with code 0
 Compose Stopping Aborting on container exit...
 Container fy0kahqxkw-algo-1-tg2wl Stopping 
 Container fy0kahqxkw-algo-1-tg2wl Stopped 


## <i class="fas fa-chart-line" style="color:#18ab4b"></i> Evaluate on held-out transactions

Unpack the `model.tar.gz` SageMaker wrote and score the test set. Fraud is rare, so precision/recall/F1 matter more than accuracy.

In [3]:
import glob
import json
import tarfile

import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

model_tar = glob.glob(os.path.join(out_dir, "**", "model.tar.gz"), recursive=True)[0]
unpacked = tempfile.mkdtemp(prefix="fraud-model-")
with tarfile.open(model_tar) as tar:
    tar.extractall(unpacked, filter="data")

model = joblib.load(os.path.join(unpacked, "fraud_model.joblib"))
features = json.load(open(os.path.join(unpacked, "model_metadata.json")))[
    "feature_names"
]

test = pd.read_csv(test_csv)
y_true = test["Class"].astype(int)
y_pred = model.predict(test[features])

print(f"fraud in test: {y_true.mean() * 100:.1f}%")
print(f"precision:     {precision_score(y_true, y_pred):.3f}")
print(f"recall:        {recall_score(y_true, y_pred):.3f}")
print(f"f1:            {f1_score(y_true, y_pred):.3f}")
print(f"accuracy:      {accuracy_score(y_true, y_pred):.3f}")

fraud in test: 20.8%
precision:     0.560
recall:        0.840
f1:            0.672
accuracy:      0.829


## <i class="fas fa-database" style="color:#18ab4b"></i> Store the trained model in oblako's S3

In [4]:
from oblako.services import S3ProxyService

s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket="fraud-models")
except Exception:
    pass
s3.upload_file(model_tar, "fraud-models", "fraud/model.tar.gz")
print("stored: s3://fraud-models/fraud/model.tar.gz")
print(
    "bucket:",
    [o["Key"] for o in s3.list_objects_v2(Bucket="fraud-models").get("Contents", [])],
)

stored: s3://fraud-models/fraud/model.tar.gz
bucket: ['fraud/model.tar.gz']
